In [ ]:
# ===== DERMNET 5-CLASS — CC + CLAHE(LAB-L) + UNSHARP(HSV-V)  =====

#
# Pipeline:
#   RGB
#   -> Shades of Gray Color Constancy
#   -> spatial transforms
#   -> CLAHE only on LAB-L
#   -> RGB
#   -> very gentle Unsharp Mask only on HSV-V
#   -> RGB
#   -> RandAugment / tensor / normalize / RandomErasing


import os
import gc
import math
import json
import copy
import zipfile
import random
from pathlib import Path
from contextlib import nullcontext

import cv2
import timm
import torch
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

from timm.data.mixup import Mixup
from timm.loss import SoftTargetCrossEntropy


# ============================================================
# 1. SEED
# ============================================================

SEED = 42


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = True


seed_everything(SEED)


# ============================================================
# 2. DEVICE & AMP
# ============================================================

USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")

scaler = torch.amp.GradScaler("cuda", enabled=USE_CUDA)


def autocast_or_noop():
    if USE_CUDA:
        return torch.amp.autocast("cuda", dtype=torch.float16)
    return nullcontext()


print(f"Device: {DEVICE} | torch: {torch.__version__} | timm: {timm.__version__}")


# ============================================================
# 3. CLASS DEFINITIONS
# ============================================================

TARGET_FOLDER_MAP = {
    "Acne and Rosacea Photos": "acne",
    "Eczema Photos": "eczema",
    "Tinea Ringworm Candidiasis and other Fungal Infections": "fungal",
    "Psoriasis pictures Lichen Planus and related diseases": "psoriasis",
}

CLASS_NAMES = ["acne", "eczema", "fungal", "psoriasis", "others"]
LABEL_MAP = {name: i for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)


# ============================================================
# 4. CONFIG
# ============================================================

class Config:
    MODEL_CANDIDATES = [
        "swinv2_small_window8_256.ms_in1k",
        "swinv2_tiny_window8_256.ms_in1k",
        "swin_base_patch4_window7_224",
        "swin_small_patch4_window7_224",
    ]

    IMG_SIZE_STAGE12 = 224
    IMG_SIZE_STAGE3 = 256

    NUM_CLASSES = NUM_CLASSES

    BATCH_SIZE = 16
    NUM_WORKERS = 2
    ACCUM_STEPS = 2

    LABEL_SMOOTHING = 0.05
    WEIGHT_DECAY = 0.03

    DROP_PATH_RATE = 0.20
    HEAD_DROPOUT = 0.25

    STAGE1_EPOCHS = 3
    STAGE2_MAX_EPOCHS = 38


    STAGE3_EPOCHS = 14
    STAGE3_WARMUP = 2

  
    MIN_DELTA_STAGE2 = 1e-4
    PATIENCE_STAGE2 = 8
    MIN_EPOCHS_STAGE2 = 10

    MIN_DELTA_STAGE3 = 2e-4
    PATIENCE_STAGE3 = 5
    MIN_EPOCHS_STAGE3 = 4

    STAGE1_BACKBONE_LR = 2e-5
    STAGE1_HEAD_LR = 8e-4

    STAGE2_LAYER3_LR = 8e-5
    STAGE2_LAYER2_LR = 4e-5
    STAGE2_HEAD_LR = 3e-4

  
    STAGE3_LR = 4.5e-6
    STAGE3_HEAD_LR = 1.2e-5

    CLIP_GRAD_NORM = 1.0

    FOCAL_GAMMA = 1.5
    EMA_DECAY = 0.9995

    OTHERS_COUNT = 2000

    # Color Constancy
    CC_POWER = 6
    CC_PERCENTILE = 99.5
    CC_EPS = 1e-6

    # CLAHE — LAB-L üzerinde
    CLAHE_CLIP_LIMIT = 0.5
    CLAHE_TILE_GRID = (8, 8)
    CLAHE_BLEND = 0.25

    # Unsharp Mask — HSV-V üzerinde çok hafif
    UNSHARP_SIGMA = 0.8
    UNSHARP_AMOUNT = 0.10
    UNSHARP_THRESHOLD = 6.0

    # TTA / calibration options
    ENABLE_VAL_TTA_SELECTION = True
    ENABLE_LOGIT_BIAS_TUNING = True

    LOGIT_BIAS_OTHERS_MAX = 0.35

    OUT_DIR = Path("/kaggle/working")
    SAVE_PREFIX = "dermnet_5class_cc_clahe_lab_unsharp_hsvv_v2_improved"


# ============================================================
# 5. DATA COLLECTION & BALANCING
# ============================================================

DATA_ROOT = None

if Path("/kaggle/input/dermnet").exists():
    DATA_ROOT = Path("/kaggle/input/dermnet")
else:
    import kagglehub
    DATA_ROOT = Path(kagglehub.dataset_download("shubhamgoel27/dermnet"))

TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"

assert TRAIN_DIR.exists() and TEST_DIR.exists(), "Dataset bulunamadı!"

IMG_EXT = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}


def collect_images(dirs):
    records = []

    for d in dirs:
        for cls_dir in sorted(d.iterdir()):
            if not cls_dir.is_dir():
                continue

            for f in cls_dir.iterdir():
                if f.is_file() and f.suffix in IMG_EXT:
                    records.append((str(f), cls_dir.name))

    return records


all_records = collect_images([TRAIN_DIR, TEST_DIR])

print(f"Toplam görsel: {len(all_records)}")

rng_data = np.random.RandomState(SEED)

target_buckets = {short: [] for short in TARGET_FOLDER_MAP.values()}
others_pool = []

for path, folder_name in all_records:
    if folder_name in TARGET_FOLDER_MAP:
        target_buckets[TARGET_FOLDER_MAP[folder_name]].append(path)
    else:
        others_pool.append(path)

print("\nTarget class sayıları:")

for k, v in target_buckets.items():
    print(f"  {k:12s}: {len(v)}")

print(f"  {'others pool':12s}: {len(others_pool)}")

others_count = min(Config.OTHERS_COUNT, len(others_pool))

print(f"\nOthers count: {others_count} (config={Config.OTHERS_COUNT})")

rng_data.shuffle(others_pool)
others_sampled = others_pool[:others_count]

all_paths = []
all_labels = []

for cls_name, paths in target_buckets.items():
    all_paths.extend(paths)
    all_labels.extend([LABEL_MAP[cls_name]] * len(paths))

all_paths.extend(others_sampled)
all_labels.extend([LABEL_MAP["others"]] * len(others_sampled))

all_paths = np.array(all_paths)
all_labels = np.array(all_labels, dtype=np.int64)

print("\nFinal dağılım:")

for i, name in enumerate(CLASS_NAMES):
    cnt = int((all_labels == i).sum())
    print(f"  {name:12s} ({i}): {cnt}")

print(f"  {'TOPLAM':12s}  : {len(all_labels)}")


# ============================================================
# 6. STRATIFIED 80/10/10 SPLIT
# ============================================================

def stratified_three_split(paths, labels, val_ratio=0.10, test_ratio=0.10, seed=42):
    rng = np.random.RandomState(seed)

    tr_idx = []
    va_idx = []
    te_idx = []

    for c in range(labels.max() + 1):
        idx = np.where(labels == c)[0]
        rng.shuffle(idx)

        n = len(idx)
        n_test = max(1, int(n * test_ratio))
        n_val = max(1, int(n * val_ratio))

        te_idx.append(idx[:n_test])
        va_idx.append(idx[n_test:n_test + n_val])
        tr_idx.append(idx[n_test + n_val:])

    tr_idx = np.concatenate(tr_idx)
    va_idx = np.concatenate(va_idx)
    te_idx = np.concatenate(te_idx)

    rng.shuffle(tr_idx)
    rng.shuffle(va_idx)
    rng.shuffle(te_idx)

    return tr_idx, va_idx, te_idx


train_idx, val_idx, test_idx = stratified_three_split(
    all_paths,
    all_labels,
    val_ratio=0.10,
    test_ratio=0.10,
    seed=SEED,
)

train_paths, train_labels = all_paths[train_idx], all_labels[train_idx]
val_paths, val_labels = all_paths[val_idx], all_labels[val_idx]
test_paths, test_labels = all_paths[test_idx], all_labels[test_idx]

print(f"\nSplit → train: {len(train_paths)} | val: {len(val_paths)} | test: {len(test_paths)}")

for split_name, lbls in [
    ("train", train_labels),
    ("val", val_labels),
    ("test", test_labels),
]:
    dist = {CLASS_NAMES[i]: int((lbls == i).sum()) for i in range(NUM_CLASSES)}
    print(f"  [{split_name}]", dist)


# ============================================================
# 7. PREPROCESSING + TRANSFORMS
# ============================================================

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

CV_RGB2LAB = getattr(cv2, "COLOR_RGB2LAB", cv2.COLOR_RGB2Lab)
CV_LAB2RGB = getattr(cv2, "COLOR_LAB2RGB", cv2.COLOR_Lab2RGB)

CV_RGB2HSV = cv2.COLOR_RGB2HSV
CV_HSV2RGB = cv2.COLOR_HSV2RGB


class ShadesOfGray:
    def __init__(self, power=6, percentile=99.5, eps=1e-6):
        self.power = power
        self.pct = percentile
        self.eps = eps

    def __call__(self, img: Image.Image) -> Image.Image:
        arr = np.asarray(img).astype(np.float32) + 1.0

        illum = np.power(
            np.mean(np.power(arr, self.power), axis=(0, 1)),
            1.0 / self.power,
        )

        illum = illum / (np.linalg.norm(illum) + self.eps)
        scale = np.sqrt(3.0) / (illum + self.eps)

        arr = arr * scale[None, None, :]

        hi = np.percentile(arr, self.pct)
        arr = np.clip(arr * (255.0 / (hi + self.eps)), 0, 255).astype(np.uint8)

        return Image.fromarray(arr)


class CLAHELab:
    """
    RGB -> LAB -> CLAHE sadece L kanalında -> original L ile blend -> RGB.
    """

    def __init__(self, clip_limit=0.5, tile_grid_size=(8, 8), blend=0.25):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size
        self.blend = blend

    def __call__(self, img: Image.Image) -> Image.Image:
        arr = np.ascontiguousarray(np.asarray(img).astype(np.uint8))

        lab = cv2.cvtColor(arr, CV_RGB2LAB)
        l, a, b = cv2.split(lab)

        l_orig = l.copy()

        clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit,
            tileGridSize=self.tile_grid_size,
        )

        l_clahe = clahe.apply(l)

        if self.blend >= 1.0:
            l_new = l_clahe
        elif self.blend <= 0.0:
            l_new = l_orig
        else:
            l_new = cv2.addWeighted(
                l_clahe,
                float(self.blend),
                l_orig,
                float(1.0 - self.blend),
                0.0,
            )

        rgb = cv2.cvtColor(cv2.merge((l_new, a, b)), CV_LAB2RGB)

        return Image.fromarray(rgb)


class UnsharpMaskHSVValue:
    """
    RGB -> HSV -> Unsharp sadece V kanalında -> RGB.

    H ve S kanallarına dokunulmaz.
    Bu sayede renk tonu/doygunluk bozulması azaltılır.
    """

    def __init__(self, sigma=0.8, amount=0.10, threshold=6.0):
        self.sigma = sigma
        self.amount = amount
        self.threshold = threshold

    def __call__(self, img: Image.Image) -> Image.Image:
        arr = np.ascontiguousarray(np.asarray(img).astype(np.uint8))

        hsv = cv2.cvtColor(arr, CV_RGB2HSV)
        h, s, v = cv2.split(hsv)

        v_f = v.astype(np.float32)

        blurred = cv2.GaussianBlur(
            v_f,
            ksize=(0, 0),
            sigmaX=self.sigma,
            sigmaY=self.sigma,
        )

        sharpened = cv2.addWeighted(
            v_f,
            1.0 + self.amount,
            blurred,
            -self.amount,
            0.0,
        )

        if self.threshold > 0:
            low_contrast_mask = np.abs(v_f - blurred) < self.threshold
            sharpened[low_contrast_mask] = v_f[low_contrast_mask]

        v_new = np.clip(sharpened, 0, 255).astype(np.uint8)

        rgb = cv2.cvtColor(cv2.merge((h, s, v_new)), CV_HSV2RGB)

        return Image.fromarray(rgb)


_cc = ShadesOfGray(
    power=Config.CC_POWER,
    percentile=Config.CC_PERCENTILE,
    eps=Config.CC_EPS,
)

_clahe_lab = CLAHELab(
    clip_limit=Config.CLAHE_CLIP_LIMIT,
    tile_grid_size=Config.CLAHE_TILE_GRID,
    blend=Config.CLAHE_BLEND,
)

_unsharp_hsv_v = UnsharpMaskHSVValue(
    sigma=Config.UNSHARP_SIGMA,
    amount=Config.UNSHARP_AMOUNT,
    threshold=Config.UNSHARP_THRESHOLD,
)


def build_transforms(img_size: int, train: bool):
    if train:
        return transforms.Compose([
            _cc,

            transforms.RandomResizedCrop(img_size, scale=(0.70, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.20),
            transforms.RandomRotation(degrees=15),

            _clahe_lab,
            _unsharp_hsv_v,

            transforms.RandAugment(num_ops=3, magnitude=12),

            transforms.ToTensor(),
            transforms.Normalize(MEAN, STD),

            transforms.RandomErasing(
                p=0.15,
                scale=(0.02, 0.15),
                ratio=(0.3, 3.3),
                value="random",
            ),
        ])

    return transforms.Compose([
        _cc,

        transforms.Resize(img_size + 32),
        transforms.CenterCrop(img_size),

        _clahe_lab,
        _unsharp_hsv_v,

        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])


# ============================================================
# 8. DATASET & LOADERS
# ============================================================

class SkinDataset(Dataset):
    def __init__(self, paths, labels, transform=None, max_retries=10):
        self.paths = paths
        self.labels = labels
        self.transform = transform
        self.max_retries = max_retries

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        for _ in range(self.max_retries):
            try:
                img = Image.open(self.paths[idx]).convert("RGB")

                if self.transform:
                    img = self.transform(img)

                y = torch.tensor(int(self.labels[idx]), dtype=torch.long)

                return img, y

            except Exception:
                idx = random.randint(0, len(self.paths) - 1)

        raise RuntimeError("Çok fazla yükleme hatası.")


def build_loaders(img_size: int):
    tr_tf = build_transforms(img_size, train=True)
    va_tf = build_transforms(img_size, train=False)

    tr_ds = SkinDataset(train_paths, train_labels, tr_tf)
    va_ds = SkinDataset(val_paths, val_labels, va_tf)
    te_ds = SkinDataset(test_paths, test_labels, va_tf)

    counts = np.bincount(train_labels, minlength=NUM_CLASSES)

    class_w = 1.0 / np.sqrt(counts + 1e-6)
    sample_w = class_w[train_labels]

    eff_unit = Config.BATCH_SIZE * Config.ACCUM_STEPS
    n_samples = max(eff_unit, (len(sample_w) // eff_unit) * eff_unit)

    sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_w, dtype=torch.double),
        num_samples=n_samples,
        replacement=True,
    )

    common_kwargs = dict(
        num_workers=Config.NUM_WORKERS,
        pin_memory=USE_CUDA,
        persistent_workers=(Config.NUM_WORKERS > 0),
    )

    tr_loader = DataLoader(
        tr_ds,
        batch_size=Config.BATCH_SIZE,
        sampler=sampler,
        drop_last=True,
        **common_kwargs,
    )

    va_loader = DataLoader(
        va_ds,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        **common_kwargs,
    )

    te_loader = DataLoader(
        te_ds,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        **common_kwargs,
    )

    return tr_loader, va_loader, te_loader


train_loader_224, val_loader_224, _ = build_loaders(Config.IMG_SIZE_STAGE12)


# ============================================================
# 9. LOSS FUNCTIONS
# ============================================================

mixup_fn = Mixup(
    mixup_alpha=0.2,
    cutmix_alpha=1.0,
    prob=0.8,
    switch_prob=0.65,
    mode="batch",
    label_smoothing=Config.LABEL_SMOOTHING,
    num_classes=NUM_CLASSES,
)

ce_soft = SoftTargetCrossEntropy()
ce_plain = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)


class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 1.5, label_smoothing: float = 0.05, num_classes: int = 5):
        super().__init__()

        self.gamma = gamma
        self.ls = label_smoothing
        self.nc = num_classes

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_p = F.log_softmax(logits, dim=1)
        p = torch.exp(log_p)

        p_t = p.gather(1, targets.view(-1, 1)).squeeze(1)
        focal_weight = (1.0 - p_t).pow(self.gamma).detach()

        smooth = self.ls / (self.nc - 1)

        log_pt = log_p.gather(1, targets.view(-1, 1)).squeeze(1)
        log_psum = log_p.sum(dim=1)

        ce = -(1.0 - self.ls) * log_pt - smooth * (log_psum - log_pt)

        return (focal_weight * ce).mean()


focal_loss = FocalLoss(
    gamma=Config.FOCAL_GAMMA,
    label_smoothing=Config.LABEL_SMOOTHING,
    num_classes=NUM_CLASSES,
)


# ============================================================
# 10. EMA
# ============================================================

class ModelEMA:
    def __init__(self, model: nn.Module, decay: float = 0.9995):
        self.decay = decay

        self.ema = copy.deepcopy(model)
        self.ema.eval()

        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        ema_sd = self.ema.state_dict()
        model_sd = model.state_dict()

        for k, ema_v in ema_sd.items():
            if k not in model_sd:
                continue

            model_v = model_sd[k].detach()

            if ema_v.shape != model_v.shape:
                continue

            model_v = model_v.to(device=ema_v.device)

            if torch.is_floating_point(ema_v):
                ema_v.mul_(self.decay).add_(
                    model_v.to(dtype=ema_v.dtype),
                    alpha=1.0 - self.decay,
                )
            else:
                ema_v.copy_(model_v)

    def state_dict(self):
        return self.ema.state_dict()


model_ema = None


# ============================================================
# 11. MODEL
# ============================================================

def pick_model(candidates):
    avail = set(timm.list_models(pretrained=True))

    for name in candidates:
        if name in avail:
            return name

    return candidates[0]


MODEL_NAME = pick_model(Config.MODEL_CANDIDATES)

print(f"Seçilen model: {MODEL_NAME}")


def create_model():
    m = timm.create_model(
        MODEL_NAME,
        pretrained=True,
        num_classes=NUM_CLASSES,
        drop_path_rate=Config.DROP_PATH_RATE,
    )

    if hasattr(m, "head") and isinstance(m.head, nn.Linear):
        in_f = m.head.in_features

        m.head = nn.Sequential(
            nn.Dropout(Config.HEAD_DROPOUT),
            nn.Linear(in_f, NUM_CLASSES),
        )

    return m


def configure_model_input_size(model, img_size: int):
    if hasattr(model, "set_input_size"):
        try:
            model.set_input_size(img_size=(img_size, img_size))
        except TypeError:
            model.set_input_size(img_size=img_size)

    if hasattr(model, "patch_embed"):
        if hasattr(model.patch_embed, "strict_img_size"):
            model.patch_embed.strict_img_size = False

        if hasattr(model.patch_embed, "img_size"):
            model.patch_embed.img_size = None


model = create_model().to(DEVICE)
configure_model_input_size(model, Config.IMG_SIZE_STAGE12)

model_ema = ModelEMA(model, decay=Config.EMA_DECAY)


# ============================================================
# 12. STAGE HELPERS
# ============================================================

def set_trainable_stage(model, stage: int):
    for p in model.parameters():
        p.requires_grad = False

    for n, p in model.named_parameters():
        if n.startswith("head.") or "norm" in n:
            p.requires_grad = True

    if stage >= 2:
        for n, p in model.named_parameters():
            if n.startswith("layers.3.") or n.startswith("layers.2."):
                p.requires_grad = True

    if stage >= 3:
        # Full unfreeze.
        for p in model.parameters():
            p.requires_grad = True

    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"[Stage {stage}] Trainable: {trainable / 1e6:.2f}M / {total / 1e6:.2f}M")


def _split_params_stage2(model):
    head = []
    l3 = []
    l2 = []
    other = []

    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue

        if n.startswith("head."):
            head.append(p)
        elif n.startswith("layers.3."):
            l3.append(p)
        elif n.startswith("layers.2."):
            l2.append(p)
        else:
            other.append(p)

    return head, l3, l2, other


def build_optimizer_stage1(model):
    head = []
    backbone = []

    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue

        if n.startswith("head."):
            head.append(p)
        else:
            backbone.append(p)

    groups = []

    if backbone:
        groups.append({"params": backbone, "lr": Config.STAGE1_BACKBONE_LR})

    if head:
        groups.append({"params": head, "lr": Config.STAGE1_HEAD_LR})

    return optim.AdamW(groups, weight_decay=Config.WEIGHT_DECAY)


def build_optimizer_stage2(model):
    head, l3, l2, other = _split_params_stage2(model)

    groups = []

    if other:
        groups.append({"params": other, "lr": Config.STAGE2_LAYER2_LR * 0.25})

    if l2:
        groups.append({"params": l2, "lr": Config.STAGE2_LAYER2_LR})

    if l3:
        groups.append({"params": l3, "lr": Config.STAGE2_LAYER3_LR})

    if head:
        groups.append({"params": head, "lr": Config.STAGE2_HEAD_LR})

    return optim.AdamW(groups, weight_decay=Config.WEIGHT_DECAY)


def build_optimizer_stage3(model):
    scale = {
        "l3": 1.00,
        "l2": 0.65,
        "l1": 0.35,
        "l0": 0.15,
        "other": 0.05,
        "head": Config.STAGE3_HEAD_LR / Config.STAGE3_LR,
    }

    buckets = {k: [] for k in scale}

    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue

        if n.startswith("head.") or n.startswith("norm."):
            buckets["head"].append(p)
        elif n.startswith("layers.3."):
            buckets["l3"].append(p)
        elif n.startswith("layers.2."):
            buckets["l2"].append(p)
        elif n.startswith("layers.1."):
            buckets["l1"].append(p)
        elif n.startswith("layers.0."):
            buckets["l0"].append(p)
        else:
            buckets["other"].append(p)

    groups = []
    log_parts = []

    for k, params in buckets.items():
        if not params:
            continue

        lr = Config.STAGE3_LR * scale[k]
        groups.append({"params": params, "lr": lr})
        log_parts.append(f"{k}:{lr:.1e}")

    print(f"  Stage3 LRs → {' | '.join(log_parts)}")

    return optim.AdamW(groups, weight_decay=Config.WEIGHT_DECAY)


# ============================================================
# 13. METRICS
# ============================================================

def confusion_matrix_np(y_true, y_pred, n):
    cm = np.zeros((n, n), dtype=np.int64)

    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1

    return cm


def metrics_from_cm(cm):
    eps = 1e-12

    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp

    prec = tp / (tp + fp + eps)
    rec = tp / (tp + fn + eps)
    f1 = 2 * prec * rec / (prec + rec + eps)

    sup = cm.sum(1).astype(np.float64)

    f1_macro = np.mean(f1)
    f1_weighted = (f1 * sup).sum() / (sup.sum() + eps)

    class_acc = tp / (sup + eps)

    return f1_macro, f1_weighted, prec, rec, f1, class_acc


# ============================================================
# 14. TRAIN / EVAL
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion, mixup=None, ema=None):
    model.train()

    optimizer.zero_grad(set_to_none=True)

    total = 0
    correct = 0
    running_loss = 0.0

    for step, (x, y) in enumerate(tqdm(loader, desc="TRAIN", leave=False)):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        use_mix = (mixup is not None) and (x.size(0) % 2 == 0)

        if use_mix:
            x, y_soft = mixup(x, y)
            hard_y = y_soft.argmax(1)
        else:
            y_soft = y
            hard_y = y

        with autocast_or_noop():
            logits = model(x)
            loss = criterion(logits, y_soft) / Config.ACCUM_STEPS

        if USE_CUDA:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        should_step = ((step + 1) % Config.ACCUM_STEPS == 0) or ((step + 1) == len(loader))

        if should_step:
            if USE_CUDA:
                scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.CLIP_GRAD_NORM)

            if USE_CUDA:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)

            if ema is not None:
                ema.update(model)

        bs = hard_y.size(0)

        running_loss += loss.item() * bs * Config.ACCUM_STEPS
        correct += (logits.argmax(1) == hard_y).sum().item()
        total += bs

    return running_loss / max(1, total), 100.0 * correct / max(1, total)


@torch.inference_mode()
def evaluate(model, loader, criterion):
    model.eval()

    total = 0
    correct = 0
    running_loss = 0.0

    all_pred = []
    all_true = []

    for x, y in tqdm(loader, desc="EVAL", leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with autocast_or_noop():
            logits = model(x)
            loss = criterion(logits, y)

        bs = y.size(0)

        running_loss += loss.item() * bs

        pred = logits.argmax(1)

        correct += (pred == y).sum().item()
        total += bs

        all_pred.append(pred.cpu().numpy())
        all_true.append(y.cpu().numpy())

    all_pred = np.concatenate(all_pred)
    all_true = np.concatenate(all_true)

    cm = confusion_matrix_np(all_true, all_pred, NUM_CLASSES)
    f1m, f1w, prec, rec, f1, acc_c = metrics_from_cm(cm)

    return running_loss / max(1, total), 100.0 * correct / max(1, total), f1m, f1w, cm


# ============================================================
# 15. PLOT HELPERS
# ============================================================

Config.OUT_DIR.mkdir(parents=True, exist_ok=True)


def plot_curves(history, out_dir):
    if not history:
        return

    epochs = [h["epoch"] for h in history]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Training Curves — CC + CLAHE(LAB-L) + Unsharp(HSV-V) v2", fontsize=13)

    axes[0].plot(epochs, [h["train_loss"] for h in history], label="train")
    axes[0].plot(epochs, [h["val_loss"] for h in history], label="val")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss")
    axes[0].legend()

    axes[1].plot(epochs, [h["train_acc"] for h in history], label="train acc")
    axes[1].plot(epochs, [h["val_acc"] for h in history], label="val acc")
    axes[1].plot(epochs, [100.0 * h["val_f1_weighted"] for h in history], label="val F1w x100")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].set_title("Accuracy / F1")
    axes[1].legend()

    plt.tight_layout()

    out_path = out_dir / f"loss_acc_curves_{Config.SAVE_PREFIX}.png"
    plt.savefig(out_path, dpi=160)
    plt.close()

    print(f"Curves saved: {out_path}")


def plot_confusion_matrix(cm, class_names, out_file, title="Confusion Matrix", normalize=True):
    cm_show = cm.astype(np.float64)

    if normalize:
        row_sum = cm_show.sum(1, keepdims=True) + 1e-12
        cm_show = cm_show / row_sum

    n = len(class_names)

    fig, ax = plt.subplots(figsize=(7, 6))

    im = ax.imshow(
        cm_show,
        interpolation="nearest",
        cmap="Blues",
        vmin=0,
        vmax=1 if normalize else None,
    )

    plt.colorbar(im, ax=ax)

    for i in range(n):
        for j in range(n):
            val = cm_show[i, j]
            raw = cm[i, j]
            color = "white" if val > 0.5 else "black"

            if normalize:
                txt = f"{val:.2f}\n({raw})"
            else:
                txt = str(raw)

            ax.text(j, i, txt, ha="center", va="center", fontsize=9, color=color)

    ticks = np.arange(n)

    ax.set_xticks(ticks)
    ax.set_xticklabels(class_names, rotation=30, ha="right", fontsize=10)

    ax.set_yticks(ticks)
    ax.set_yticklabels(class_names, fontsize=10)

    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title + (" (normalized)" if normalize else " (raw counts)"))

    plt.tight_layout()
    plt.savefig(out_file, dpi=180)
    plt.close()


def print_and_save_numeric_cm(cm, class_names, out_dir, split_name="test"):
    print(f"\n{'=' * 60}")
    print(f"SAYISAL CONFUSION MATRIX — {split_name.upper()}")
    print(f"{'=' * 60}")

    col_w = 12

    header = f"{'True\\Pred':>12}" + "".join(f"{n:>{col_w}}" for n in class_names)

    print(header)
    print("-" * len(header))

    for i, row_name in enumerate(class_names):
        row = f"{row_name:>12}" + "".join(
            f"{cm[i, j]:>{col_w}}" for j in range(len(class_names))
        )
        print(row)

    f1m, f1w, prec, rec, f1, acc_c = metrics_from_cm(cm)

    print(f"\n{'Class':12s} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    print("-" * 52)

    for i, name in enumerate(class_names):
        sup = int(cm[i].sum())
        print(f"{name:12s} {prec[i]:>10.4f} {rec[i]:>10.4f} {f1[i]:>10.4f} {sup:>10d}")

    print("-" * 52)
    print(f"{'macro':12s} {prec.mean():>10.4f} {rec.mean():>10.4f} {f1m:>10.4f}")
    print(f"{'weighted':12s} {'':>10} {'':>10} {f1w:>10.4f}")

    overall_acc = 100.0 * np.diag(cm).sum() / cm.sum()

    print(f"\nOverall Accuracy: {overall_acc:.2f}%")
    print(f"F1 Macro:         {f1m:.4f}")
    print(f"F1 Weighted:      {f1w:.4f}")

    df = pd.DataFrame(cm, index=class_names, columns=class_names)

    csv_path = out_dir / f"cm_{split_name}_raw_{Config.SAVE_PREFIX}.csv"
    df.to_csv(csv_path)

    print(f"\nCSV kaydedildi: {csv_path}")

    return overall_acc, f1m, f1w


# ============================================================
# 16. STAGE RUNNER — F1w + val_loss tie-breaker
# ============================================================

history = []

best_val_f1w = 0.0
best_val_loss_at_best = float("inf")
best_state = None
best_epoch_num = -1

global_epoch = 0


def run_stage(
    stage_id,
    train_loader,
    val_loader,
    optimizer_builder,
    epochs,
    use_mixup,
    min_delta,
    patience,
    min_epochs,
    set_input_size_to=None,
    warmup_epochs=0,
    criterion_override=None,
):
    global model
    global model_ema
    global best_val_f1w
    global best_val_loss_at_best
    global best_state
    global best_epoch_num
    global global_epoch
    global history

    if set_input_size_to is not None:
        configure_model_input_size(model, set_input_size_to)

    set_trainable_stage(model, stage_id)

    optimizer = optimizer_builder(model)

    if warmup_epochs > 0:
        def lr_lambda(ep):
            if ep < warmup_epochs:
                return float(ep + 1) / float(warmup_epochs)

            cos_prog = float(ep - warmup_epochs) / float(max(1, epochs - warmup_epochs))

            return 0.5 * (1.0 + math.cos(math.pi * cos_prog))

        scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

    else:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=epochs,
            eta_min=0.0,
        )

    mix = mixup_fn if use_mixup else None

    if criterion_override is not None:
        crit = criterion_override
    else:
        crit = ce_soft if use_mixup else ce_plain

    no_improve = 0

    for e in range(epochs):
        tr_loss, tr_acc = train_one_epoch(
            model,
            train_loader,
            optimizer,
            crit,
            mixup=mix,
            ema=model_ema,
        )

        va_loss, va_acc, va_f1m, va_f1w, _ = evaluate(
            model,
            val_loader,
            ce_plain,
        )

        global_epoch += 1

        lrs = [pg["lr"] for pg in optimizer.param_groups]

        history.append({
            "epoch": global_epoch,
            "stage": stage_id,
            "train_loss": float(tr_loss),
            "train_acc": float(tr_acc),
            "val_loss": float(va_loss),
            "val_acc": float(va_acc),
            "val_f1_macro": float(va_f1m),
            "val_f1_weighted": float(va_f1w),
            "lrs": lrs,
        })

        print(
            f"Epoch {global_epoch:03d} | Stage {stage_id} | "
            f"train {tr_loss:.4f}/{tr_acc:.2f}% | "
            f"val {va_loss:.4f}/{va_acc:.2f}% | "
            f"F1w {va_f1w:.4f} | F1m {va_f1m:.4f}"
        )

        # F1w ana metrik.
        # F1 çok yakınsa daha düşük val_loss olan modeli seç.
        is_f1_better = va_f1w > best_val_f1w + min_delta

        is_f1_tie_but_loss_better = (
            abs(float(va_f1w) - float(best_val_f1w)) <= min_delta
            and float(va_loss) < float(best_val_loss_at_best)
        )

        if is_f1_better or is_f1_tie_but_loss_better:
            best_val_f1w = float(va_f1w)
            best_val_loss_at_best = float(va_loss)
            best_state = copy.deepcopy(model.state_dict())
            best_epoch_num = global_epoch
            no_improve = 0

            reason = "F1w" if is_f1_better else "F1w tie + lower val_loss"

            print(
                f"  → BEST by {reason}: "
                f"val_f1w={best_val_f1w:.4f}, "
                f"val_loss={best_val_loss_at_best:.4f} "
                f"@ epoch {best_epoch_num}"
            )

        else:
            no_improve += 1
            print(f"  → no improve ({no_improve}/{patience})")

        scheduler.step()

        if e + 1 >= min_epochs and no_improve >= patience:
            print(f"[Stage {stage_id}] Early stop at local epoch {e + 1}")
            break


# ============================================================
# 17. TRAINING — 3 STAGE
# ============================================================

print("\n" + "=" * 60)
print("TRAINING")
print("=" * 60)

# ---------------------------
# Stage 1
# ---------------------------
run_stage(
    stage_id=1,
    train_loader=train_loader_224,
    val_loader=val_loader_224,
    optimizer_builder=build_optimizer_stage1,
    epochs=Config.STAGE1_EPOCHS,
    use_mixup=True,
    min_delta=1e-4,
    patience=99,
    min_epochs=99,
    set_input_size_to=Config.IMG_SIZE_STAGE12,
)

# ---------------------------
# Stage 2
# ---------------------------
run_stage(
    stage_id=2,
    train_loader=train_loader_224,
    val_loader=val_loader_224,
    optimizer_builder=build_optimizer_stage2,
    epochs=Config.STAGE2_MAX_EPOCHS,
    use_mixup=True,
    min_delta=Config.MIN_DELTA_STAGE2,
    patience=Config.PATIENCE_STAGE2,
    min_epochs=Config.MIN_EPOCHS_STAGE2,
    set_input_size_to=Config.IMG_SIZE_STAGE12,
)

# ---------------------------
# Build Stage 3 loaders
# ---------------------------
train_loader_256, val_loader_256, test_loader_256 = build_loaders(Config.IMG_SIZE_STAGE3)

# Load best Stage 1/2 state.
if best_state is not None:
    model.load_state_dict(best_state)

configure_model_input_size(model, Config.IMG_SIZE_STAGE3)

# Restart EMA from 256-ready model.
model_ema = ModelEMA(model, decay=Config.EMA_DECAY)

# Baseline @256.
base_val_loss, base_val_acc, base_f1m, base_f1w, _ = evaluate(
    model,
    val_loader_256,
    ce_plain,
)

best_val_f1w = float(base_f1w)
best_val_loss_at_best = float(base_val_loss)
best_state = copy.deepcopy(model.state_dict())
best_epoch_num = global_epoch

print("\n" + "-" * 60)
print(
    f"[Stage 3 baseline @256] "
    f"val_loss={base_val_loss:.4f} | "
    f"val_acc={base_val_acc:.2f}% | "
    f"F1w={base_f1w:.4f} | "
    f"F1m={base_f1m:.4f}"
)
print("-" * 60)

# ---------------------------
# Stage 3
# ---------------------------
run_stage(
    stage_id=3,
    train_loader=train_loader_256,
    val_loader=val_loader_256,
    optimizer_builder=build_optimizer_stage3,
    epochs=Config.STAGE3_EPOCHS,
    use_mixup=False,
    min_delta=Config.MIN_DELTA_STAGE3,
    patience=Config.PATIENCE_STAGE3,
    min_epochs=Config.MIN_EPOCHS_STAGE3,
    set_input_size_to=Config.IMG_SIZE_STAGE3,
    warmup_epochs=Config.STAGE3_WARMUP,
    criterion_override=focal_loss,
)

if best_state is not None:
    model.load_state_dict(best_state)

configure_model_input_size(model, Config.IMG_SIZE_STAGE3)


# ============================================================
# 18. EMA EVALUATION — F1w first, loss tie-breaker
# ============================================================

print("\n" + "=" * 60)
print("EMA DEĞERLENDİRME")
print("=" * 60)

reg_val_loss, reg_val_acc, reg_f1m, reg_f1w, _ = evaluate(
    model,
    val_loader_256,
    ce_plain,
)

print(
    f"Regular best : "
    f"loss={reg_val_loss:.4f} | "
    f"acc={reg_val_acc:.2f}% | "
    f"F1w={reg_f1w:.4f} | "
    f"F1m={reg_f1m:.4f}"
)

configure_model_input_size(model_ema.ema, Config.IMG_SIZE_STAGE3)

ema_val_loss, ema_val_acc, ema_f1m, ema_f1w, _ = evaluate(
    model_ema.ema,
    val_loader_256,
    ce_plain,
)

print(
    f"EMA model    : "
    f"loss={ema_val_loss:.4f} | "
    f"acc={ema_val_acc:.2f}% | "
    f"F1w={ema_f1w:.4f} | "
    f"F1m={ema_f1m:.4f}"
)

ema_used = False

ema_better = (
    ema_f1w > reg_f1w + 1e-4
    or (
        abs(float(ema_f1w) - float(reg_f1w)) <= 1e-4
        and float(ema_val_loss) < float(reg_val_loss)
    )
)

if ema_better:
    print("✓ EMA daha iyi. EMA ağırlıkları kullanılacak.")
    model.load_state_dict(model_ema.state_dict())
    configure_model_input_size(model, Config.IMG_SIZE_STAGE3)

    best_val_f1w = float(ema_f1w)
    best_val_loss_at_best = float(ema_val_loss)
    best_state = copy.deepcopy(model.state_dict())
    ema_used = True
else:
    print("✗ Regular best daha iyi. Orijinal ağırlıklar korunuyor.")

del model_ema
gc.collect()

if USE_CUDA:
    torch.cuda.empty_cache()


# ============================================================
# 19. TTA HELPERS
# ============================================================

@torch.inference_mode()
def predict_tta(
    model,
    loader,
    use_softmax_average=False,
    logit_bias=None,
    desc="TEST(TTA-5)",
):
    model.eval()

    all_pred = []
    all_true = []

    if logit_bias is not None:
        if isinstance(logit_bias, np.ndarray):
            logit_bias = torch.tensor(logit_bias, dtype=torch.float32)
        logit_bias = logit_bias.to(DEVICE).view(1, -1)

    for x, y in tqdm(loader, desc=desc, leave=False):
        x = x.to(DEVICE, non_blocking=True)

        with autocast_or_noop():
            logits_list = [
                model(x),
                model(torch.flip(x, dims=[3])),
                model(torch.flip(x, dims=[2])),
                model(torch.rot90(x, k=1, dims=[2, 3])),
                model(torch.rot90(x, k=3, dims=[2, 3])),
            ]

            if logit_bias is not None:
                logits_list = [z + logit_bias for z in logits_list]

            if use_softmax_average:
                probs = sum(torch.softmax(z, dim=1) for z in logits_list) / len(logits_list)
                pred = probs.argmax(1)
            else:
                logits = sum(logits_list) / len(logits_list)
                pred = logits.argmax(1)

        all_pred.append(pred.cpu().numpy())
        all_true.append(y.numpy())

    all_pred = np.concatenate(all_pred)
    all_true = np.concatenate(all_true)

    cm = confusion_matrix_np(all_true, all_pred, NUM_CLASSES)

    acc = 100.0 * (all_true == all_pred).mean()
    f1m, f1w, *_ = metrics_from_cm(cm)

    return acc, f1m, f1w, cm


@torch.inference_mode()
def collect_tta_logits(model, loader, desc="COLLECT LOGITS(TTA-5)"):
    model.eval()

    all_logits = []
    all_true = []

    for x, y in tqdm(loader, desc=desc, leave=False):
        x = x.to(DEVICE, non_blocking=True)

        with autocast_or_noop():
            logits_list = [
                model(x),
                model(torch.flip(x, dims=[3])),
                model(torch.flip(x, dims=[2])),
                model(torch.rot90(x, k=1, dims=[2, 3])),
                model(torch.rot90(x, k=3, dims=[2, 3])),
            ]

            logits = sum(logits_list) / len(logits_list)

        all_logits.append(logits.float().cpu())
        all_true.append(y.cpu())

    return torch.cat(all_logits, dim=0), torch.cat(all_true, dim=0)


def tune_simple_logit_bias_on_val(model, val_loader):
    """
    Küçük post-hoc bias tuning.
    Amaç:
      - Özellikle others recall'u artırmak.
      - Psoriasis'e fazla kaçan others örneklerini azaltmak.
    """

    val_logits, val_y = collect_tta_logits(model, val_loader)

    val_logits_np = val_logits.numpy()
    val_y_np = val_y.numpy()

    best = {
        "acc": -1.0,
        "f1w": -1.0,
        "f1m": -1.0,
        "bias": np.zeros(NUM_CLASSES, dtype=np.float32),
        "cm": None,
    }


    others_grid = np.linspace(0.00, Config.LOGIT_BIAS_OTHERS_MAX, 8)
    psoriasis_grid = np.linspace(-0.20, 0.05, 6)
    fungal_grid = np.linspace(-0.08, 0.08, 5)
    eczema_grid = np.linspace(-0.08, 0.08, 5)

    for b_others in others_grid:
        for b_psoriasis in psoriasis_grid:
            for b_fungal in fungal_grid:
                for b_eczema in eczema_grid:
                    bias = np.zeros(NUM_CLASSES, dtype=np.float32)

                    bias[LABEL_MAP["others"]] = float(b_others)
                    bias[LABEL_MAP["psoriasis"]] = float(b_psoriasis)
                    bias[LABEL_MAP["fungal"]] = float(b_fungal)
                    bias[LABEL_MAP["eczema"]] = float(b_eczema)

                    pred = (val_logits_np + bias[None, :]).argmax(axis=1)

                    cm = confusion_matrix_np(val_y_np, pred, NUM_CLASSES)
                    acc = 100.0 * (pred == val_y_np).mean()
                    f1m, f1w, *_ = metrics_from_cm(cm)

                    better = (
                        acc > best["acc"] + 1e-9
                        or (
                            abs(acc - best["acc"]) <= 1e-9
                            and f1w > best["f1w"] + 1e-12
                        )
                        or (
                            abs(acc - best["acc"]) <= 1e-9
                            and abs(f1w - best["f1w"]) <= 1e-12
                            and f1m > best["f1m"]
                        )
                    )

                    if better:
                        best = {
                            "acc": float(acc),
                            "f1w": float(f1w),
                            "f1m": float(f1m),
                            "bias": bias.copy(),
                            "cm": cm.copy(),
                        }

    print("\n" + "=" * 60)
    print("VALIDATION LOGIT BIAS TUNING")
    print("=" * 60)
    print(
        f"Best val ACC: {best['acc']:.2f}% | "
        f"F1w: {best['f1w']:.4f} | "
        f"F1m: {best['f1m']:.4f}"
    )
    print("Best bias:", {CLASS_NAMES[i]: float(best["bias"][i]) for i in range(NUM_CLASSES)})

    return best["bias"], best


def select_tta_mode_on_val(model, val_loader):
    """
    Validation üzerinde logit-average ve softmax-average TTA'yı karşılaştırır.
    Öncelik accuracy, sonra F1w.
    """

    print("\n" + "=" * 60)
    print("VALIDATION TTA MODE SELECTION")
    print("=" * 60)

    acc_l, f1m_l, f1w_l, cm_l = predict_tta(
        model,
        val_loader,
        use_softmax_average=False,
        logit_bias=None,
        desc="VAL TTA logits",
    )

    print(f"Val TTA logits  : acc={acc_l:.2f}% | F1w={f1w_l:.4f} | F1m={f1m_l:.4f}")

    acc_s, f1m_s, f1w_s, cm_s = predict_tta(
        model,
        val_loader,
        use_softmax_average=True,
        logit_bias=None,
        desc="VAL TTA softmax",
    )

    print(f"Val TTA softmax : acc={acc_s:.2f}% | F1w={f1w_s:.4f} | F1m={f1m_s:.4f}")

    softmax_better = (
        acc_s > acc_l + 1e-9
        or (
            abs(acc_s - acc_l) <= 1e-9
            and f1w_s > f1w_l
        )
    )

    if softmax_better:
        print("✓ Softmax-average TTA seçildi.")
        return True, {
            "mode": "softmax_average",
            "val_acc": float(acc_s),
            "val_f1w": float(f1w_s),
            "val_f1m": float(f1m_s),
            "cm": cm_s,
        }

    print("✓ Logit-average TTA seçildi.")
    return False, {
        "mode": "logit_average",
        "val_acc": float(acc_l),
        "val_f1w": float(f1w_l),
        "val_f1m": float(f1m_l),
        "cm": cm_l,
    }


# ============================================================
# 20. VALIDATION TTA SELECTION + OPTIONAL BIAS
# ============================================================

configure_model_input_size(model, Config.IMG_SIZE_STAGE3)

if Config.ENABLE_VAL_TTA_SELECTION:
    use_softmax_average, tta_val_info = select_tta_mode_on_val(model, val_loader_256)
else:
    use_softmax_average = False
    tta_val_info = {
        "mode": "logit_average",
        "val_acc": None,
        "val_f1w": None,
        "val_f1m": None,
        "cm": None,
    }

best_logit_bias = None
bias_info = None

if Config.ENABLE_LOGIT_BIAS_TUNING and not use_softmax_average:
    best_logit_bias, bias_info = tune_simple_logit_bias_on_val(model, val_loader_256)
else:
    print("\nLogit bias tuning atlandı.")
    if use_softmax_average:
        print("Sebep: softmax-average TTA seçildi; bias logit-space için tasarlandı.")
    else:
        print("Sebep: Config.ENABLE_LOGIT_BIAS_TUNING=False")


# ============================================================
# 21. TEST — TTA-5
# ============================================================

print("\n" + "=" * 60)
print("FINAL TEST")
print("=" * 60)

test_acc, test_f1m, test_f1w, test_cm = predict_tta(
    model,
    test_loader_256,
    use_softmax_average=use_softmax_average,
    logit_bias=best_logit_bias,
    desc="TEST(TTA-5)",
)

print("\n" + "=" * 70)
print(f"BEST EPOCH      : {best_epoch_num}")
print(f"BEST VAL F1w    : {best_val_f1w:.4f}")
print(f"BEST VAL LOSS   : {best_val_loss_at_best:.4f}")
print(f"EMA USED        : {ema_used}")
print(f"TTA MODE        : {'softmax_average' if use_softmax_average else 'logit_average'}")
print(f"LOGIT BIAS USED : {best_logit_bias is not None}")
if best_logit_bias is not None:
    print("LOGIT BIAS      :", {CLASS_NAMES[i]: float(best_logit_bias[i]) for i in range(NUM_CLASSES)})
print(f"TEST ACC (TTA-5): {test_acc:.2f}%  |  F1w: {test_f1w:.4f}  |  F1m: {test_f1m:.4f}")
print("=" * 70)


# ============================================================
# 22. NUMERIC CONFUSION MATRIX
# ============================================================


val_acc_final, val_f1m_final, val_f1w_final, val_cm_final = predict_tta(
    model,
    val_loader_256,
    use_softmax_average=use_softmax_average,
    logit_bias=best_logit_bias,
    desc="VAL FINAL(TTA-5)",
)

val_acc_cm, val_f1m_cm, val_f1w_cm = print_and_save_numeric_cm(
    val_cm_final,
    CLASS_NAMES,
    Config.OUT_DIR,
    split_name="val",
)

test_acc_num, test_f1m_num, test_f1w_num = print_and_save_numeric_cm(
    test_cm,
    CLASS_NAMES,
    Config.OUT_DIR,
    split_name="test",
)


# ============================================================
# 23. VISUALS
# ============================================================

plot_curves(history, Config.OUT_DIR)

plot_confusion_matrix(
    val_cm_final,
    CLASS_NAMES,
    Config.OUT_DIR / f"cm_val_normalized_{Config.SAVE_PREFIX}.png",
    title="Val CM — CC+CLAHE(LAB-L)+Unsharp(HSV-V) v2",
    normalize=True,
)

plot_confusion_matrix(
    test_cm,
    CLASS_NAMES,
    Config.OUT_DIR / f"cm_test_normalized_{Config.SAVE_PREFIX}.png",
    title="Test CM — CC+CLAHE(LAB-L)+Unsharp(HSV-V) v2",
    normalize=True,
)

plot_confusion_matrix(
    val_cm_final,
    CLASS_NAMES,
    Config.OUT_DIR / f"cm_val_raw_{Config.SAVE_PREFIX}.png",
    title="Val CM — CC+CLAHE(LAB-L)+Unsharp(HSV-V) v2 raw",
    normalize=False,
)

plot_confusion_matrix(
    test_cm,
    CLASS_NAMES,
    Config.OUT_DIR / f"cm_test_raw_{Config.SAVE_PREFIX}.png",
    title="Test CM — CC+CLAHE(LAB-L)+Unsharp(HSV-V) v2 raw",
    normalize=False,
)

print("Tüm görseller kaydedildi.")


# ============================================================
# 24. CHECKPOINT
# ============================================================

ckpt_path = Config.OUT_DIR / f"{Config.SAVE_PREFIX}_best.pth"

payload = {
    "model_name": MODEL_NAME,
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "label_map": LABEL_MAP,

    "best_epoch": int(best_epoch_num),
    "best_val_f1w": float(best_val_f1w),
    "best_val_loss_at_best": float(best_val_loss_at_best),

    "ema_used": bool(ema_used),

    "tta_mode": "softmax_average" if use_softmax_average else "logit_average",
    "tta_val_info": {
        "mode": tta_val_info["mode"],
        "val_acc": tta_val_info["val_acc"],
        "val_f1w": tta_val_info["val_f1w"],
        "val_f1m": tta_val_info["val_f1m"],
    },

    "logit_bias_used": best_logit_bias is not None,
    "logit_bias": None if best_logit_bias is None else [float(x) for x in best_logit_bias],

    "test_acc_tta5": float(test_acc),
    "test_f1w": float(test_f1w),
    "test_f1m": float(test_f1m),

    "val_acc_final_tta5": float(val_acc_final),
    "val_f1w_final_tta5": float(val_f1w_final),
    "val_f1m_final_tta5": float(val_f1m_final),

    "history": history,
    "state_dict": model.state_dict(),

    "img_size_stage12": Config.IMG_SIZE_STAGE12,
    "img_size_stage3": Config.IMG_SIZE_STAGE3,

    "preprocessing": {
        "pipeline": [
            {
                "type": "ShadesOfGray",
                "power": Config.CC_POWER,
                "percentile": Config.CC_PERCENTILE,
                "eps": Config.CC_EPS,
            },
            {
                "type": "CLAHE_LAB_L",
                "clip_limit": Config.CLAHE_CLIP_LIMIT,
                "tile_grid": Config.CLAHE_TILE_GRID,
                "blend": Config.CLAHE_BLEND,
            },
            {
                "type": "UNSHARP_HSV_V",
                "sigma": Config.UNSHARP_SIGMA,
                "amount": Config.UNSHARP_AMOUNT,
                "threshold": Config.UNSHARP_THRESHOLD,
            },
        ],
    },

    "training_changes_vs_83_50_base": {
        "stage2_min_delta": Config.MIN_DELTA_STAGE2,
        "stage2_patience": Config.PATIENCE_STAGE2,
        "stage3_epochs": Config.STAGE3_EPOCHS,
        "stage3_warmup": Config.STAGE3_WARMUP,
        "stage3_lr": Config.STAGE3_LR,
        "stage3_head_lr": Config.STAGE3_HEAD_LR,
        "stage3_lr_scale": {
            "l3": 1.00,
            "l2": 0.65,
            "l1": 0.35,
            "l0": 0.15,
            "other": 0.05,
            "head": Config.STAGE3_HEAD_LR / Config.STAGE3_LR,
        },
        "best_selection": "val_f1_weighted, tie-breaker lower val_loss",
        "tta_selection": "validation-selected logit-average vs softmax-average",
        "logit_bias_tuning": bool(Config.ENABLE_LOGIT_BIAS_TUNING),
    },

    "version": "cc_clahe_lab_unsharp_hsvv_v2_improved",
    "seed": SEED,

    "split_info": {
        "n_train": int(len(train_paths)),
        "n_val": int(len(val_paths)),
        "n_test": int(len(test_paths)),
    },
}

torch.save(payload, ckpt_path)

print(f"Checkpoint kaydedildi: {ckpt_path}")


# ============================================================
# 25. SUMMARY JSON
# ============================================================

summary = {
    "version": "CC_CLAHE_LAB_UNSHARP_HSVV_V2_IMPROVED",
    "model_name": MODEL_NAME,
    "seed": SEED,

    "best_epoch": int(best_epoch_num),
    "best_val_f1w": float(best_val_f1w),
    "best_val_loss_at_best": float(best_val_loss_at_best),

    "ema_used": bool(ema_used),

    "tta_mode": "softmax_average" if use_softmax_average else "logit_average",
    "logit_bias_used": best_logit_bias is not None,
    "logit_bias": None if best_logit_bias is None else {
        CLASS_NAMES[i]: float(best_logit_bias[i]) for i in range(NUM_CLASSES)
    },

    "test_acc_tta5": float(test_acc),
    "test_f1_weighted": float(test_f1w),
    "test_f1_macro": float(test_f1m),

    "val_acc_final_tta5": float(val_acc_final),
    "val_f1_weighted_final_tta5": float(val_f1w_final),
    "val_f1_macro_final_tta5": float(val_f1m_final),

    "class_names": CLASS_NAMES,

    "n_train": int(len(train_paths)),
    "n_val": int(len(val_paths)),
    "n_test": int(len(test_paths)),

    "preprocessing": {
        "color_constancy": {
            "method": "ShadesOfGray",
            "power": Config.CC_POWER,
            "percentile": Config.CC_PERCENTILE,
            "eps": Config.CC_EPS,
        },
        "clahe": {
            "space": "LAB",
            "channel": "L",
            "clip_limit": Config.CLAHE_CLIP_LIMIT,
            "tile_grid": Config.CLAHE_TILE_GRID,
            "blend": Config.CLAHE_BLEND,
        },
        "unsharp": {
            "space": "HSV",
            "channel": "V",
            "sigma": Config.UNSHARP_SIGMA,
            "amount": Config.UNSHARP_AMOUNT,
            "threshold": Config.UNSHARP_THRESHOLD,
        },
    },

    "changes_vs_83_50_base": {
        "stage2_min_delta": "1e-3 -> 1e-4",
        "stage2_patience": "6 -> 8",
        "stage3_epochs": "22 -> 14",
        "stage3_warmup": "3 -> 2",
        "stage3_lr": "7e-6 -> 4.5e-6",
        "stage3_head_lr": "1.5e-5 -> 1.2e-5",
        "stage3_early_layer_lr": "reduced",
        "best_selection": "val F1w with lower val_loss tie-breaker",
        "tta": "validation-selected logits/softmax TTA",
        "optional_logit_bias": "small validation-tuned bias for others confusion",
    },
}

summary_path = Config.OUT_DIR / f"summary_{Config.SAVE_PREFIX}.json"

with open(summary_path, "w", encoding="utf-8") as fp:
    json.dump(summary, fp, indent=2, ensure_ascii=False)

print(f"Özet JSON: {summary_path}")


# ============================================================
# 26. ZIP
# ============================================================

zip_path = Config.OUT_DIR / f"{Config.SAVE_PREFIX}_outputs.zip"

files_to_zip = [
    ckpt_path,
    summary_path,

    Config.OUT_DIR / f"loss_acc_curves_{Config.SAVE_PREFIX}.png",

    Config.OUT_DIR / f"cm_val_normalized_{Config.SAVE_PREFIX}.png",
    Config.OUT_DIR / f"cm_test_normalized_{Config.SAVE_PREFIX}.png",

    Config.OUT_DIR / f"cm_val_raw_{Config.SAVE_PREFIX}.png",
    Config.OUT_DIR / f"cm_test_raw_{Config.SAVE_PREFIX}.png",

    Config.OUT_DIR / f"cm_val_raw_{Config.SAVE_PREFIX}.csv",
    Config.OUT_DIR / f"cm_test_raw_{Config.SAVE_PREFIX}.csv",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in files_to_zip:
        f = Path(f)

        if f.exists():
            z.write(f, arcname=f.name)
        else:
            print(f"  [WARN] bulunamadı, atlandı: {f}")

print(f"\nZIP kaydedildi: {zip_path}")
print("Kaggle Output panelinden indirebilirsin →", zip_path.name)

Device: cuda | torch: 2.10.0+cu128 | timm: 1.0.25
Toplam görsel: 19559

Target class sayıları:
  acne        : 1152
  eczema      : 1544
  fungal      : 1625
  psoriasis   : 1757
  others pool : 13481

Others count: 2000 (config=2000)

Final dağılım:
  acne         (0): 1152
  eczema       (1): 1544
  fungal       (2): 1625
  psoriasis    (3): 1757
  others       (4): 2000
  TOPLAM        : 8078

Split → train: 6466 | val: 806 | test: 806
  [train] {'acne': 922, 'eczema': 1236, 'fungal': 1301, 'psoriasis': 1407, 'others': 1600}
  [val] {'acne': 115, 'eczema': 154, 'fungal': 162, 'psoriasis': 175, 'others': 200}
  [test] {'acne': 115, 'eczema': 154, 'fungal': 162, 'psoriasis': 175, 'others': 200}
Seçilen model: swinv2_small_window8_256.ms_in1k


model.safetensors:   0%|          | 0.00/201M [00:00<?, ?B/s]


TRAINING
[Stage 1] Trainable: 0.04M / 48.96M


Epoch 001 | Stage 1 | train 1.4996/35.69% | val 1.3944/44.79% | F1w 0.4280 | F1m 0.4363
  → BEST by F1w: val_f1w=0.4280, val_loss=1.3944 @ epoch 1


Epoch 002 | Stage 1 | train 1.4281/42.85% | val 1.3569/44.91% | F1w 0.4324 | F1m 0.4403
  → BEST by F1w: val_f1w=0.4324, val_loss=1.3569 @ epoch 2


...

EMA model    : loss=0.7353 | acc=79.90% | F1w=0.7975 | F1m=0.8088
✗ Regular best daha iyi. Orijinal ağırlıklar korunuyor.

VALIDATION TTA MODE SELECTION


Val TTA logits  : acc=80.77% | F1w=0.8058 | F1m=0.8154


Val TTA softmax : acc=81.02% | F1w=0.8085 | F1m=0.8179
✓ Softmax-average TTA seçildi.

Logit bias tuning atlandı.
Sebep: softmax-average TTA seçildi; bias logit-space için tasarlandı.

FINAL TEST



BEST EPOCH      : 44
BEST VAL F1w    : 0.8100
BEST VAL LOSS   : 0.7366
EMA USED        : False
TTA MODE        : softmax_average
LOGIT BIAS USED : False
TEST ACC (TTA-5): 84.00%  |  F1w: 0.8396  |  F1m: 0.8468



SAYISAL CONFUSION MATRIX — VAL
   True\Pred        acne      eczema      fungal   psoriasis      others
------------------------------------------------------------------------
        acne         106           2           0           3           4
      eczema           0         131           6           8           9
      fungal           2          10         131           5          14
   psoriasis           0           8           4         150          13
      others          12          16          12          25         135

Class         Precision     Recall         F1    Support
----------------------------------------------------
acne             0.8833     0.9217     0.9021        115
eczema           0.7844     0.8506     0.8162        154
fungal           0.8562     0.8086     0.8317        162
psoriasis        0.7853     0.8571     0.8197        175
others           0.7714     0.6750     0.7200        200
----------------------------------------------------
macro   

In [3]:
2

2

In [4]:
5

5

In [5]:
1

1

In [6]:
23

23